# Multi-Stage Pipeline — Platform Setup

This is Stage 0 of the multi-stage training pipeline: the **platform
team's** one-time work. Attendees typically run this notebook once
and never again; `advanced_multistage_training_pipeline.ipynb` then consumes
the Registry entries it produces.

**What this notebook does**

1. Creates two Registries:
   - `airfoil_datasets` — catalogue for the pipeline's dataset assets.
   - `airfoil_models` — catalogue for the pipeline's model assets.
2. Publishes an **un-trained** `TinyTransolver` to
   `airfoil_models/base:production`. This is the shared architecture
   every downstream team starts from.
3. Generates and promotes three purpose-built datasets to
   `airfoil_datasets/{synthetic, pretraining, posttraining}:production`
   — meaningfully different in case count, AoA window, and Re window.

Everything happens inside one `wandb.init(job_type="publish-base")`
run so the platform team's contribution shows up as one lineage node
in the graph.

**Prerequisites**
- `uv sync`, `uv run wandb login`.
- Access to the `wandb_emea` entity — or edit `ENTITY` in §1.
- No other notebooks required.

**Next step:** once this notebook completes, run
`advanced_multistage_training_pipeline.ipynb` to walk through the three
training stages that consume these Registry entries.


## 1. Setup

Third-party imports and constants. Both Registry names are declared
here so the publishing cell can reference them.


In [1]:
from __future__ import annotations

import json
import math
import tempfile
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import wandb

device = torch.device("cpu")

ENTITY = "wandb_emea"
PROJECT = "jb_physics_demo"

DATASETS_REG = "airfoil_datasets"
MODELS_REG = "airfoil_models"


def utc_ts() -> str:
    return datetime.now(timezone.utc).isoformat()


print(f"torch      = {torch.__version__}")
print(f"wandb      = {wandb.__version__}")
print(f"Registries = {DATASETS_REG}, {MODELS_REG}")


torch      = 2.12.1
wandb      = 0.28.0
Registries = airfoil_datasets, airfoil_models


## 2. Utilities — model class + synthetic generator

Inline everything the platform team needs: the shared
`TinyTransolver` architecture (so we can save its un-trained
weights), the `AirfoilCase` dataclass, the parameterised NACA
generator (so we can produce three genuinely different datasets from
one code path), and a JSON serialiser.

Read this once and skim past — the interesting story is in §3.


In [2]:
# ------------------- Model -------------------
class PhysicsAttention(nn.Module):
    def __init__(self, hidden: int, n_slices: int, n_heads: int, ffn_expansion: int = 2):
        super().__init__()
        assert hidden % n_heads == 0
        self.n_slices = n_slices
        self.n_heads = n_heads
        self.head_dim = hidden // n_heads

        self.slice_proj = nn.Linear(hidden, n_slices)
        self.qkv = nn.Linear(hidden, 3 * hidden)
        self.attn_out = nn.Linear(hidden, hidden)
        self.ffn = nn.Sequential(
            nn.Linear(hidden, ffn_expansion * hidden),
            nn.GELU(),
            nn.Linear(ffn_expansion * hidden, hidden),
        )
        self.norm1 = nn.LayerNorm(hidden)
        self.norm2 = nn.LayerNorm(hidden)

    def _slice_attend_deslice(self, x: torch.Tensor) -> torch.Tensor:
        B, N, C = x.shape
        M, H, D = self.n_slices, self.n_heads, self.head_dim
        w = torch.softmax(self.slice_proj(x), dim=-1)
        numer = torch.einsum("bnm,bnc->bmc", w, x)
        denom = w.sum(dim=1).unsqueeze(-1) + 1e-8
        z = numer / denom
        qkv = self.qkv(z).view(B, M, 3, H, D).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(D)
        attn = torch.softmax(scores, dim=-1)
        z_out = (attn @ v).transpose(1, 2).contiguous().view(B, M, C)
        z_out = self.attn_out(z_out)
        return torch.einsum("bnm,bmc->bnc", w, z_out)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self._slice_attend_deslice(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class TinyTransolver(nn.Module):
    def __init__(
        self,
        in_dim: int = 5, out_dim: int = 3, hidden: int = 128,
        n_layers: int = 6, n_slices: int = 32, n_heads: int = 8,
        ffn_expansion: int = 2,
    ):
        super().__init__()
        self.embed = nn.Linear(in_dim, hidden)
        self.blocks = nn.ModuleList([
            PhysicsAttention(hidden, n_slices, n_heads, ffn_expansion)
            for _ in range(n_layers)
        ])
        self.out_norm = nn.LayerNorm(hidden)
        self.head = nn.Linear(hidden, out_dim)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        x = self.embed(features)
        for block in self.blocks:
            x = block(x)
        return self.head(self.out_norm(x))


MODEL_CONFIG = dict(
    in_dim=5, out_dim=3, hidden=128, n_layers=6,
    n_slices=32, n_heads=8, ffn_expansion=2,
)


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ------------------- Data -------------------
@dataclass
class AirfoilCase:
    points: np.ndarray
    sdf: np.ndarray
    aoa: float
    re: float
    pressure: np.ndarray
    velocity: np.ndarray


def _naca4_camber(x: np.ndarray, m: float, p: float) -> np.ndarray:
    yc = np.zeros_like(x)
    if p > 0:
        mask = x < p
        yc[mask] = (m / p ** 2) * (2 * p * x[mask] - x[mask] ** 2)
        yc[~mask] = (m / (1 - p) ** 2) * ((1 - 2 * p) + 2 * p * x[~mask] - x[~mask] ** 2)
    return yc


def _naca4_thickness(x: np.ndarray, t: float) -> np.ndarray:
    return (t / 0.2) * (
        0.2969 * np.sqrt(np.clip(x, 1e-6, None))
        - 0.1260 * x
        - 0.3516 * x ** 2
        + 0.2843 * x ** 3
        - 0.1015 * x ** 4
    )


def _sample_case_ranged(
    rng: np.random.Generator, n_points: int,
    aoa_lo_deg: float, aoa_hi_deg: float,
    re_lo: float, re_hi: float,
) -> AirfoilCase:
    m = rng.uniform(0.0, 0.06)
    p = rng.uniform(0.2, 0.6)
    t = rng.uniform(0.08, 0.18)
    aoa = rng.uniform(aoa_lo_deg, aoa_hi_deg) * np.pi / 180.0
    re = 10 ** rng.uniform(np.log10(re_lo), np.log10(re_hi))

    pts = rng.uniform([-0.5, -0.8], [1.5, 0.8], size=(n_points, 2)).astype(np.float32)

    x_chord = np.linspace(0.01, 1.0, 200)
    yc = _naca4_camber(x_chord, m, p)
    yt = _naca4_thickness(x_chord, t)
    surface = np.concatenate(
        [np.stack([x_chord, yc + yt], axis=1),
         np.stack([x_chord, yc - yt], axis=1)[::-1]],
        axis=0,
    )
    dists = np.linalg.norm(pts[:, None, :] - surface[None, :, :], axis=-1)
    sdf = dists.min(axis=1).astype(np.float32)

    r = np.linalg.norm(pts - np.array([0.5, 0.0]), axis=1)
    r = np.maximum(r, 0.1)
    theta = np.arctan2(pts[:, 1], pts[:, 0] - 0.5)
    u_inf, v_inf = np.cos(aoa), np.sin(aoa)
    circulation = -2 * np.pi * np.sin(aoa) * (1.0 + m * 5.0)
    doublet = 0.25 * t
    u = np.clip(
        u_inf - doublet * np.cos(2 * theta) / r ** 2
        - circulation * np.sin(theta) / (2 * np.pi * r),
        -3.0, 3.0,
    )
    v = np.clip(
        v_inf - doublet * np.sin(2 * theta) / r ** 2
        + circulation * np.cos(theta) / (2 * np.pi * r),
        -3.0, 3.0,
    )
    pressure = np.clip(1.0 - (u ** 2 + v ** 2), -3.0, 1.0)
    return AirfoilCase(
        points=pts,
        sdf=sdf,
        aoa=float(aoa),
        re=float(re),
        pressure=pressure.astype(np.float32),
        velocity=np.stack([u, v], axis=1).astype(np.float32),
    )


def synthetic_dataset_ranged(
    n_cases: int, n_points: int,
    aoa_lo_deg: float, aoa_hi_deg: float,
    re_lo: float, re_hi: float,
    seed: int,
) -> list[AirfoilCase]:
    rng = np.random.default_rng(seed)
    return [
        _sample_case_ranged(rng, n_points, aoa_lo_deg, aoa_hi_deg, re_lo, re_hi)
        for _ in range(n_cases)
    ]


def serialise_cases_to_json(cases: list[AirfoilCase], path: Path, meta: dict) -> None:
    payload = {
        "meta": meta,
        "cases": [
            {
                "aoa": float(c.aoa), "re": float(c.re),
                "points": c.points.tolist(), "sdf": c.sdf.tolist(),
                "pressure": c.pressure.tolist(), "velocity": c.velocity.tolist(),
            }
            for c in cases
        ],
    }
    path.write_text(json.dumps(payload))


print(f"model params = {count_params(TinyTransolver()):,}")


model params = 821,059


## 3. Publish the base model and the three datasets

One `wandb.init(job_type="publish-base")` run does everything:

1. Creates both Registries via `wandb.Api().create_registry(...)`,
   wrapped in `try/except` so this cell is idempotent — safe to
   re-run.
2. Instantiates a random-init `TinyTransolver` and writes its
   `state_dict` + architecture config to a checkpoint file. Logs it
   as `airfoil_base_model` (a project artifact of type `"model"`),
   then `link_artifact`s the version into
   `airfoil_models/base` with aliases `production` + `v1`.
3. Generates three datasets from `synthetic_dataset_ranged`, each
   with distinct parameters:

   | Collection | Cases × Points | AoA (deg) | Re | Purpose |
   | ---------- | -------------- | --------- | -- | ------- |
   | `synthetic` | 20 × 256 | [-4, 8] | [1e6, 3e6] | Small, cheap — warm init only |
   | `pretraining` | 200 × 512 | [-12, 15] | [1e6, 1e7] | Wide flight envelope — bulk pretraining |
   | `posttraining` | 40 × 512 | [10, 15] | [3e6, 6e6] | Narrow high-lift regime — task-specific fine-tune |

   Each dataset is logged as a JSON artifact and linked into its own
   Collection under `airfoil_datasets`.

Result: four `:production`-aliased entries — 1 model + 3 datasets —
ready for the three training-stage teams to consume.


In [3]:
DATASET_SPECS = [
    ("synthetic",    dict(n_cases=20,  n_points=256,
                          aoa_lo_deg=-4.0,  aoa_hi_deg=8.0,
                          re_lo=1e6, re_hi=3e6, seed=100)),
    ("pretraining",  dict(n_cases=200, n_points=512,
                          aoa_lo_deg=-12.0, aoa_hi_deg=15.0,
                          re_lo=1e6, re_hi=1e7, seed=200)),
    ("posttraining", dict(n_cases=40,  n_points=512,
                          aoa_lo_deg=10.0,  aoa_hi_deg=15.0,
                          re_lo=3e6, re_hi=6e6, seed=300)),
]

tmp_dir = Path(tempfile.mkdtemp())

with wandb.init(
    entity=ENTITY, project=PROJECT,
    job_type="publish-base",
    name="publish-base-and-datasets",
    tags=["notebook", "multistage", "platform"],
) as run:
    api = wandb.Api()

    # 1. Ensure both Registries exist (idempotent).
    for reg in (DATASETS_REG, MODELS_REG):
        try:
            api.create_registry(name=reg, visibility="organization")
            print(f"created registry: {reg}")
        except Exception as exc:
            print(f"registry {reg} create skipped: {exc}")

    # 2. Publish the un-trained base model.
    base_model = TinyTransolver(**MODEL_CONFIG).to(device)
    base_path = tmp_dir / "base_model.pt"
    torch.save(
        {
            "state_dict": base_model.state_dict(),
            "config": dict(MODEL_CONFIG),
            "stage": "base",
        },
        base_path,
    )
    base_art = wandb.Artifact(
        name="airfoil_base_model",
        type="model",
        description="Un-trained TinyTransolver — starting point for staged training.",
        metadata={
            "stage": "base",
            "n_params": count_params(base_model),
            **MODEL_CONFIG,
            "created_at": utc_ts(),
        },
    )
    base_art.add_file(str(base_path), name="checkpoint.pt")
    logged = run.log_artifact(base_art)
    logged.wait()
    run.link_artifact(
        logged,
        target_path=f"wandb-registry-{MODELS_REG}/base",
        aliases=["production", "v1"],
    )
    print(f"published base model -> wandb-registry-{MODELS_REG}/base:production")

    # 3. Generate + promote the three datasets.
    for name, cfg in DATASET_SPECS:
        cases = synthetic_dataset_ranged(**cfg)
        path = tmp_dir / f"{name}.json"
        serialise_cases_to_json(
            cases, path,
            meta={"name": name, **cfg,
                  "n_cases_generated": len(cases),
                  "created_at": utc_ts()},
        )
        size_kb = path.stat().st_size / 1024
        art = wandb.Artifact(
            name=f"airfoil_dataset_{name}",
            type="dataset",
            description=f"{name} dataset for the multi-stage pipeline.",
            metadata={
                "stage": name, **cfg,
                "n_cases_generated": len(cases),
                "size_kb": size_kb,
                "created_at": utc_ts(),
            },
        )
        art.add_file(str(path), name=f"{name}.json")
        logged = run.log_artifact(art)
        logged.wait()
        run.link_artifact(
            logged,
            target_path=f"wandb-registry-{DATASETS_REG}/{name}",
            aliases=["production", "v1"],
        )
        print(f"published {name} dataset ({len(cases)} cases, {size_kb:.1f} KB) "
              f"-> wandb-registry-{DATASETS_REG}/{name}:production")


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: jbutt (wandb_emea) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


created registry: airfoil_datasets
created registry: airfoil_models
published base model -> wandb-registry-airfoil_models/base:production
published synthetic dataset (20 cases, 641.7 KB) -> wandb-registry-airfoil_datasets/synthetic:production
published pretraining dataset (200 cases, 12756.2 KB) -> wandb-registry-airfoil_datasets/pretraining:production
published posttraining dataset (40 cases, 2532.1 KB) -> wandb-registry-airfoil_datasets/posttraining:production


## 4. Wrap up — what to look at next

**In the W&B UI**, browse to the org-level Registry page and you
should see:

- `airfoil_datasets` with three Collections (`synthetic`,
  `pretraining`, `posttraining`), each with one `:production`
  version.
- `airfoil_models` with one Collection (`base`), one `:production`
  version.

**Next step.** Run `notebooks/multistage_training_pipeline.ipynb`.
That notebook picks up from here: three teams consume the Registry
entries this notebook produced, each running one training stage
(init → pretrain → posttrain), promoting their output into the
`initialised`, `pretrained`, and `posttrained` Collections
respectively.

This platform-setup notebook is intentionally idempotent — safe to
re-run if you want to bump dataset versions or refresh the base
model. Each re-run produces a new `:v2`, `:v3`, ... version; the
`production` alias is moved to point at the newest.
